# detach-stop-gradient-trick — worked example 1: Detach G's output so the D-step leaves G's gradients at zero

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-stop-gradient-trick`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

In the GAN discriminator step you score fake images from the generator. If you feed `G(z)` directly, `loss.backward()` flows all the way back into G's parameters — polluting G with a gradient that belongs to D's objective. Calling `G(z).detach()` severs the autograd graph at that point so backward stops at the fake tensor and only D gets updated.

## Worked solution

**Goal:** run a discriminator-step loss and prove that G's parameters receive no gradient.

1. **Build tiny G and D.** A single `nn.Linear` each is enough to observe gradient flow — the idiom is identical at scale.
2. **Forward through G, then detach.** `fake = G(z).detach()` produces the same numbers as `G(z)` but the returned tensor has `requires_grad=False` and `grad_fn=None`. This is the stop-gradient.
3. **Compute D's loss on detached fake vs real.** `loss = (D(fake) - D(x_real)).mean()`. Because `fake` carries no graph, the only path back from `loss` runs through D's parameters and through `x_real`'s branch of D — never through G.
4. **Backward.** `loss.backward()` accumulates gradient into D's parameters only.
5. **Verify.** Every parameter of G has `.grad is None` (it was never touched). We print whether any G grad is non-None to confirm the stop-gradient held.

Why it works: `.detach()` returns a new leaf-like tensor that shares storage but is disconnected from the computation history. Autograd cannot traverse past a node with no `grad_fn`, so the backward sweep terminates before reaching G.

In [ ]:
import torch.nn as nn

t.manual_seed(0)
G = nn.Linear(4, 4)
D = nn.Linear(4, 1)

def d_step_detached(G, D, z, x_real):
    fake = G(z).detach()
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item()

z = t.randn(8, 4)
x_real = t.randn(8, 4)
for p in list(G.parameters()) + list(D.parameters()):
    p.grad = None

loss_val = d_step_detached(G, D, z, x_real)
g_touched = any(p.grad is not None for p in G.parameters())
d_touched = all(p.grad is not None for p in D.parameters())
print('loss:', round(loss_val, 5))
print('any G grad non-None:', g_touched)
print('all D grads present:', d_touched)